# Bring Your Own Model with Amazon SageMaker AI: Script Mode in SDK v3

This notebook walks through two end-to-end examples demonstrating **script mode** in the SageMaker Python SDK v3:

1. **Example 1** — Train and deploy a scikit-learn Random Forest (tabular ML → real-time endpoint)
2. **Example 2** — Fine-tune Stable Diffusion 3.5 with LoRA (generative AI, multi-GPU distributed training)

Both examples use the same two core classes:
- `ModelTrainer` — configures and launches a SageMaker training job
- `ModelBuilder` — packages your inference handler and deploys to an endpoint

**Key concept:** The `SourceCode` object syncs a local code directory into the container at runtime — your code runs inside the container without being baked into the image.

---
## Prerequisites

- An AWS account with Amazon SageMaker AI access
- An IAM execution role with SageMaker and S3 permissions
- SageMaker Python SDK v3 installed
- Training container images pushed to Amazon ECR (see container build sections below)
- An S3 bucket for training data and model artifacts
- (Optional) An MLflow tracking server on SageMaker for experiment tracking

In [ ]:
# Install/upgrade dependencies
!pip install "sagemaker>=3.0" boto3 mlflow sagemaker-mlflow

---
## Container Setup (Prerequisites)

Before running the training examples, you need to build and push Docker images to ECR.
These containers are intentionally minimal — they contain only runtime/framework libraries,
**no training code**. All algorithm-specific code is injected at runtime via `SourceCode`.

### Scikit-learn Container

**Dockerfile** (`docker/sklearn/Dockerfile.sklearn`):
```dockerfile
FROM python:3.13-slim

RUN apt-get update && apt-get install -y \
    build-essential jq git \
    && rm -rf /var/lib/apt/lists/*

COPY docker/sklearn/requirements.txt .
RUN pip install -r requirements.txt --no-cache-dir
```

**requirements.txt:**
```
scikit-learn>=1.8.0
pandas>=2.0.0
matplotlib>=3.6.1
mlflow>=3.10.0
sagemaker-mlflow>=0.2.0
```

### Stable Diffusion Container

**Dockerfile** (`docker/stable_diffusion/Dockerfile.stablediffusion`):
```dockerfile
FROM pytorch/pytorch:2.7.1-cuda12.8-cudnn9-devel

RUN apt-get update && apt-get install -y \
    build-essential jq git \
    && rm -rf /var/lib/apt/lists/*

COPY docker/pytorch/requirements.txt .
RUN pip install -r requirements.txt --no-cache-dir
```

### Build & Push to ECR

Run these from a terminal in the project root (requires Docker installed locally):

```bash
# Sklearn container
bash build.sh --env .env.docker.sklearn
bash push.sh --env .env.docker.sklearn

# Stable Diffusion container
bash build.sh --env .env.docker.stablediffusion
bash push.sh --env .env.docker.stablediffusion
```

The `build.sh` and `push.sh` scripts handle ECR authentication, repository creation, tagging, and pushing automatically.

---
# Example 1: Train and Deploy a Scikit-learn Model

We train a Random Forest regressor on the diabetes dataset and deploy it to a real-time SageMaker endpoint.

## Step 1: Configuration

In [ ]:
import os

import boto3
from sagemaker.core.helper.session_helper import Session
from sagemaker.core.training.configs import (
    Compute,
    InputData,
    OutputDataConfig,
    SourceCode,
    StoppingCondition,
)
from sagemaker.serve.model_builder import ModelBuilder
from sagemaker.serve.mode.function_pointers import Mode
from sagemaker.serve.utils.types import ModelServer
from sagemaker.train.model_trainer import ModelTrainer

# ============================================================
# Auto-detect account context
# ============================================================
boto_session = boto3.Session()
sm_session = Session(boto_session=boto_session)

AWS_REGION = boto_session.region_name
AWS_ACCOUNT_ID = boto3.client("sts").get_caller_identity()["Account"]
SAGEMAKER_EXECUTION_ROLE = sm_session.get_caller_identity_arn()  # works for role or user
S3_BUCKET = sm_session.default_bucket()

print(f"Account : {AWS_ACCOUNT_ID}")
print(f"Region  : {AWS_REGION}")
print(f"Role    : {SAGEMAKER_EXECUTION_ROLE}")
print(f"Bucket  : {S3_BUCKET}")

In [ ]:
# ============================================================
# Training configuration
# ============================================================

# Your custom training image pushed to ECR (built in the Container Setup step
# above). Alternatively, you can point this at a prebuilt AWS Deep Learning
# Container instead of building your own — see the full list of training and
# inference DLC image URIs at:
# https://aws.github.io/deep-learning-containers/reference/available_images/
TRAINING_IMAGE_URI = f"{AWS_ACCOUNT_ID}.dkr.ecr.{AWS_REGION}.amazonaws.com/sklearn:latest"

# S3 paths
TRAINING_DATA_S3_PATH = f"s3://{S3_BUCKET}/random-forest/data"
MODEL_OUTPUT_S3_PATH = f"s3://{S3_BUCKET}/random-forest/model-output"

# MLflow on SageMaker (set to None to skip experiment tracking)
MLFLOW_ARN = "arn:aws:sagemaker:us-east-2:026898548254:mlflow-app/app-QVUHEV7ZKNXS"               # e.g. "arn:aws:sagemaker:us-east-1:123456789012:mlflow-app/app-XXXXX"
MLFLOW_EXPERIMENT_NAME = "random-forest-experiment"

# Job name
JOB_NAME = "random-forest-training"

## Step 2: Launch a Training Job with ModelTrainer

The `SourceCode` object takes your local `source_dir` and a `command` string. At job launch, SageMaker syncs the entire `source_dir` into the container and runs your command. This decouples your code from your container image — change the script, re-launch. No container rebuild needed.

In [ ]:
source_code = SourceCode(
    source_dir="./train/random_forest",
    command=(
        "python random_forest.py"
        " --n_jobs 4 --max_depth 10 --n_estimators 120"
        f" --mlflow_arn {MLFLOW_ARN}"
        f" --mlflow_experiment_name {MLFLOW_EXPERIMENT_NAME}"
    ),
)

compute = Compute(
    instance_type="ml.m5.2xlarge",
    instance_count=1,
    volume_size_in_gb=30,
    keep_alive_period_in_seconds=3600,  # Warm pool for faster re-runs
)

stopping_condition = StoppingCondition(max_runtime_in_seconds=3600)
output_config = OutputDataConfig(s3_output_path=MODEL_OUTPUT_S3_PATH)

# Note: no input_data_config — the training script fetches the Pima Indians
# Diabetes dataset directly from OpenML at runtime, so no S3 input channel
# is needed.
model_trainer = ModelTrainer(
    training_image=TRAINING_IMAGE_URI,
    source_code=source_code,
    compute=compute,
    output_data_config=output_config,
    stopping_condition=stopping_condition,
    role=SAGEMAKER_EXECUTION_ROLE,
    base_job_name=JOB_NAME,
    sagemaker_session=Session(),
)

print(f"Starting training job: {JOB_NAME}")
model_trainer.train(wait=True)
training_job_name = model_trainer._latest_training_job.training_job_name
print(f"Training complete: {training_job_name}")

## Step 3: Locate the Trained Model Artifact

In [ ]:
sm_session = Session()
sm_client = sm_session.boto_session.client("sagemaker")
training_job_desc = sm_client.describe_training_job(TrainingJobName=training_job_name)
model_artifact_s3_uri = training_job_desc["ModelArtifacts"]["S3ModelArtifacts"]
print(f"Model artifact: {model_artifact_s3_uri}")

## Step 4: Deploy to a Real-time Endpoint with ModelBuilder

ModelBuilder packages your inference handler, repacks it with the model artifact, and creates the endpoint. The same `SourceCode` pattern is used: point at a local directory containing your handler and specify the `entry_script`.

In [ ]:
# ============================================================
# Inference configuration
# ============================================================
# For deployment we use a prebuilt AWS Deep Learning Container (the DJL
# Serving inference image) — no build needed. The URI is region-dynamic.
# Browse all prebuilt training and inference DLC images here:
# https://aws.github.io/deep-learning-containers/reference/available_images/
INFERENCE_IMAGE_URI = (
    f"763104351884.dkr.ecr.{AWS_REGION}.amazonaws.com/djl-inference:0.36.0-cpu-full"
)
ENDPOINT_NAME = "random-forest-endpoint"

inference_source_code = SourceCode(
    source_dir="./deploy/random_forest",
    entry_script="inference.py",
)

model_builder = ModelBuilder(
    image_uri=INFERENCE_IMAGE_URI,
    model_server=ModelServer.DJL_SERVING,
    source_code=inference_source_code,
    s3_model_data_url=model_artifact_s3_uri,
    env_vars={"OPTION_ENTRYPOINT": "code/inference.py"},
    sagemaker_session=Session(),
    role_arn=SAGEMAKER_EXECUTION_ROLE,
)

# Workaround for v3 SDK quirk
model_builder.model_path = "/tmp/sagemaker/model-builder"
os.makedirs(model_builder.model_path, exist_ok=True)
model_builder.dependencies = []

print(f"Building model package: {ENDPOINT_NAME}")
model_builder.build(model_name=ENDPOINT_NAME, mode=Mode.SAGEMAKER_ENDPOINT)

print(f"Deploying endpoint: {ENDPOINT_NAME}")
predictor = model_builder.deploy(
    endpoint_name=ENDPOINT_NAME,
    initial_instance_count=1,
)
print(f"Endpoint ready: {ENDPOINT_NAME}")

## Step 5: Test the Endpoint

In [ ]:
# Pima Indians Diabetes dataset: 8 numeric features
# (preg, plas, pres, skin, insu, mass, pedi, age)
sample_csv = "6,148,72,35,0,33.6,0.627,50"

# v3: model_builder.deploy() returns a sagemaker.core.resources.Endpoint
# object — call .invoke() directly instead of using a boto3 runtime client.
response = predictor.invoke(
    body=sample_csv.encode("utf-8"),
    content_type="text/csv",
)

# response is an InvokeEndpointOutput; .body is a botocore StreamingBody
print("Prediction:", response.body.read().decode("utf-8"))

# The model returns a class label: 1 = tested_positive (diabetes), 0 = tested_negative.
# This sample is the first row of the Pima dataset (true label: positive), so a prediction
# of [1] confirms the endpoint is serving the trained classifier correctly.

---
# Example 2: Fine-tune Stable Diffusion 3.5 with LoRA

This example demonstrates the same `ModelTrainer` + `SourceCode` pattern for a sophisticated generative AI training job. We fine-tune Stable Diffusion 3.5 Medium using LoRA with Hugging Face Accelerate for multi-GPU distributed training on `ml.g5.12xlarge` (4× A10G GPUs).

The source directory structure:
```
train/stable_diffusion/
├── base.sh                         # Launcher: pip installs, GPU detection, accelerate launch
├── train_text_to_image_lora.py     # Main training script
├── requirements.txt                # Runtime deps installed by base.sh
├── recipes/
│   └── default-medium-g5_12x.yaml  # Hyperparameter recipe
└── accelerate_configs/
    └── ddp.yaml                    # Distributed training config
```

Swap recipes, adjust LoRA rank, or change the base model ID with a one-line YAML edit — no Docker rebuild required.

## Step 1: Configuration

In [ ]:
# ============================================================
# EDIT THESE VALUES FOR YOUR ACCOUNT
# ============================================================
SD_TRAINING_IMAGE_URI = f"{AWS_ACCOUNT_ID}.dkr.ecr.{AWS_REGION}.amazonaws.com/stablediffusion:latest"

SD_S3_BUCKET = S3_BUCKET  # Reuse the same bucket or set a different one
SD_TRAINING_DATA_S3_PATH = f"s3://{SD_S3_BUCKET}/stable-diffusion/data"
SD_MODEL_OUTPUT_S3_PATH = f"s3://{SD_S3_BUCKET}/stable-diffusion/model-output"

SD_JOB_NAME = "stable-diffusion-training"

# Secrets for gated model access and experiment tracking
# SD 3.5 Medium is a gated HF repo — you need a token with access granted
HF_TOKEN = "<YOUR_HF_TOKEN>"                   # https://huggingface.co/settings/tokens
WANDB_API_KEY = "<YOUR_WANDB_API_KEY>"          # Optional: for W&B logging

## Step 2: Launch Training Job

The `command` launches `base.sh` — a bash launcher that detects the GPU count, runs `accelerate launch`, and kicks off LoRA fine-tuning. All hyperparameters live in the YAML recipe file.

In [ ]:
sd_source_code = SourceCode(
    source_dir="./train/stable_diffusion",
    command=(
        "/bin/bash base.sh"
        " --config recipes/default-medium-g5_12x.yaml"
        " --training-script train_text_to_image_lora.py"
        " --accelerate-config accelerate_configs/ddp.yaml"
    ),
)

sd_compute = Compute(
    instance_type="ml.g5.12xlarge",  # 4x A10G GPUs
    instance_count=1,
    volume_size_in_gb=30,
    keep_alive_period_in_seconds=3600,
)

sd_stopping_condition = StoppingCondition(max_runtime_in_seconds=24 * 3600)
sd_output_config = OutputDataConfig(s3_output_path=SD_MODEL_OUTPUT_S3_PATH)

sd_input_config = [
    InputData(channel_name="train", data_source=SD_TRAINING_DATA_S3_PATH),
]

sd_model_trainer = ModelTrainer(
    training_image=SD_TRAINING_IMAGE_URI,
    source_code=sd_source_code,
    compute=sd_compute,
    output_data_config=sd_output_config,
    stopping_condition=sd_stopping_condition,
    role=SAGEMAKER_EXECUTION_ROLE,
    environment={"HF_TOKEN": HF_TOKEN, "WANDB_API_KEY": WANDB_API_KEY},
    base_job_name=SD_JOB_NAME,
    sagemaker_session=Session(),
)

print(f"Starting training job: {SD_JOB_NAME}")
sd_model_trainer.train(input_data_config=sd_input_config, wait=True)
sd_training_job_name = sd_model_trainer._latest_training_job.training_job_name
print(f"Training complete: {sd_training_job_name}")

## Step 3: Locate the Trained LoRA Weights

In [ ]:
sd_sm_session = Session()
sd_sm_client = sd_sm_session.boto_session.client("sagemaker")
sd_training_job_desc = sd_sm_client.describe_training_job(TrainingJobName=sd_training_job_name)
sd_model_artifact_s3_uri = sd_training_job_desc["ModelArtifacts"]["S3ModelArtifacts"]
print(f"LoRA weights: {sd_model_artifact_s3_uri}")

> **Next step — Deployment:** To deploy the fine-tuned Stable Diffusion model to a real-time endpoint, you would follow the same `ModelBuilder` + `SourceCode` pattern shown in Example 1, using an LMI/DJL inference container (e.g., `763104351884.dkr.ecr.us-west-2.amazonaws.com/djl-inference:0.36.0-lmi15.0.0-cu126`) and a custom inference handler that loads the LoRA weights on top of the base SD 3.5 model.

---
# Clean Up

Delete resources to avoid ongoing charges.

In [ ]:
from sagemaker.core.resources import Endpoint, EndpointConfig, Model

# Delete the sklearn endpoint, its endpoint config, and the model using the
# SDK v3 resource classes: fetch each resource with .get(), then .delete().
# (ModelBuilder names all three after ENDPOINT_NAME.)
Endpoint.get(endpoint_name=ENDPOINT_NAME).delete()
EndpointConfig.get(endpoint_config_name=ENDPOINT_NAME).delete()
Model.get(model_name=ENDPOINT_NAME).delete()

print(f"Deleted endpoint: {ENDPOINT_NAME}")
print("\nRemember to also clean up:")
print("  - S3 training data and model artifacts (if no longer needed)")
print("  - ECR container images (if no longer needed)")
print("  - MLflow tracking server (if created for this walkthrough)")